In [1]:
import os
import shutil
import pandas as pd
from tqdm import tqdm

# -------------------------
# Paths 
# -------------------------
CSV_PATH = r"archive\ISIC_2017_GroundTruth.csv"
IMAGE_DIR = r"archive\images\images"
OUTPUT_DIR = "binary_isic_2017"

# -------------------------
# Create output folders
# -------------------------
benign_dir = os.path.join(OUTPUT_DIR, "benign")
malignant_dir = os.path.join(OUTPUT_DIR, "malignant")

os.makedirs(benign_dir, exist_ok=True)
os.makedirs(malignant_dir, exist_ok=True)

# -------------------------
# Load CSV
# -------------------------
df = pd.read_csv(CSV_PATH)

# -------------------------
# Convert to Binary Class
# -------------------------
def to_binary_class(row):
    return "malignant" if row["melanoma"] == 1 else "benign"

df["binary_class"] = df.apply(to_binary_class, axis=1)

# -------------------------
# Process Images (WITH tqdm)
# -------------------------
missing_images = []

for _, row in tqdm(df.iterrows(),
                    total=len(df),
                    desc="Processing images",
                    unit="img"):
    
    image_id = row["image_id"]
    binary_class = row["binary_class"]

    image_path = None
    for ext in [".jpg", ".jpeg", ".png"]:
        p = os.path.join(IMAGE_DIR, image_id + ext)
        if os.path.exists(p):
            image_path = p
            break

    if image_path is None:
        missing_images.append(image_id)
        continue

    dest = benign_dir if binary_class == "benign" else malignant_dir
    shutil.copy(image_path, os.path.join(dest, os.path.basename(image_path)))

# -------------------------
# Save Binary CSV
# -------------------------
binary_csv = df[["image_id", "binary_class"]]
binary_csv.to_csv(os.path.join(OUTPUT_DIR, "binary_labels.csv"), index=False)

# -------------------------
# Summary
# -------------------------
print("\n✅ Binary dataset created successfully!")
print(df["binary_class"].value_counts())

if missing_images:
    print(f"\n⚠ Missing images: {len(missing_images)}")

Processing images: 100%|██████████| 2000/2000 [03:47<00:00,  8.77img/s]



✅ Binary dataset created successfully!
binary_class
benign       1626
malignant     374
Name: count, dtype: int64
